# 07. Zonal Statistics
Calculate statistics (mean, min, max, etc.) for specific polygonal regions.

In [1]:
import curaster
import json
import os

input_file = "../build/benchmark_data/test_S_2048x2048.tif"
info = curaster.open(input_file).get_info()

## Create Multiple Polygons
We will define two simple polygons.

In [2]:
gt = info["geotransform"]
w, h = info["width"], info["height"]
x_mid = gt[0] + gt[1] * (w / 2)
y_mid = gt[3] + gt[5] * (h / 2)

# zonal_stats expects a raw OGR geometry (Polygon, MultiPolygon, or GeometryCollection)
# not a FeatureCollection. Use GeometryCollection with two Polygon zones.
aoi = json.dumps({
    "type": "GeometryCollection",
    "geometries": [
        {"type": "Polygon", "coordinates": [[
            [gt[0], gt[3]], [x_mid, gt[3]], [x_mid, y_mid], [gt[0], y_mid], [gt[0], gt[3]]
        ]]},
        {"type": "Polygon", "coordinates": [[
            [x_mid, y_mid], [gt[0] + w*gt[1], y_mid], [gt[0] + w*gt[1], gt[3] + h*gt[5]], [x_mid, gt[3] + h*gt[5]], [x_mid, y_mid]
        ]]}
    ]
})

## Calculate Zonal Stats
We calculate mean, max, and sum for these zones.

In [3]:
%%time
results = curaster.open(input_file).zonal_stats(
    aoi, 
    stats=["mean", "max", "sum", "count"], 
    band=1,
    verbose=True
)

import pandas as pd
df = pd.DataFrame([r.to_dict() for r in results])
print("\nZonal Statistics Results:")
print(df)

0...10...20...30...40...50...60...70...80...90...100 - done.

Zonal Statistics Results:
   zone_id         mean    min     max      std_dev  count          sum
0        1  4152.543087  308.0  7989.0  1726.380546  33792  140322736.0
1        2  4151.769531  307.0  7984.0  1698.133022  32768  136045184.0
CPU times: user 676 ms, sys: 253 ms, total: 929 ms
Wall time: 951 ms
